# 1.0 Importando pacotes

In [1]:
"""
Este arquivo produz as variáveis:
    - roads  --> gdf de vias com geometria e valores de ADT
    - industrial_gdf  --> gdf de industrias
    - stations --> gdf de estações de monitoramento com geometria e poluentes
"""

# Pacotes e funções
import geopandas as gpd
from pathlib import Path
from long_2_utm_zone import long_2_utm_zone
from utm_zone_2_epsg import utm_zone_2_epsg
import pandas as pd
import numpy as np
import os
import sys

# Desativando notação científica
pd.set_option('display.float_format', '{:.2f}'.format)

# 2.0 Definindo caminhos

In [2]:
# Caminho da pasta de inputs
inputs_path = Path('./inputs').resolve()

# Caminho da pasta de outputs
outputs_path = Path('./outputs').resolve()

# Arquivo de todas as vias do BR
roads_path = inputs_path / 'processed_roads_dissolved.parquet'

# Arquivo de códigos de vias e os respecitvos fluxo médios diários em um
# buffer de 250 metros das estações
flow_path = inputs_path / 'stations_may_jun_2025.parquet'

# Arquivo de indústrias BR (Gerais, mineração e aterros)
industrial_path = Path('../../../../InvFontesFixas_book/dados/emission_total_light.csv').resolve()

# Planilha de estações de monitoramento do BR
stations_path = Path('../../../data/Monitoramento_QAr_BR.csv').resolve()


# 3.0 Geodataframes de Vias do Brasil

## 3.1 Lendo arquivo do modelo estático de vias

In [3]:
"""Lendo geodataframe de vias, com as colunas obrigatórias:
    'osm_id': int de código de identificação de vias do OpenStreetMaps
    'geometry': LineString de geometria da via
"""
roads = (gpd
         .read_parquet(path=roads_path)
         .reset_index(drop=False)
         .astype({'osm_id': int}))

roads.head()

,osm_id,geometry,name,highway,lanes,oneway,surface,maxspeed
0,4217292,"MULTILINESTRING ((-43.20285 -22.98436, -43.202...",Rua Vinícius de Moraes,residential,2,yes,asphalt,<NA>
1,4217293,"MULTILINESTRING ((-43.20511 -22.98637, -43.205...",Rua Joana Angélica,residential,2,yes,asphalt,<NA>
2,4217297,"MULTILINESTRING ((-43.20679 -22.98073, -43.206...",Rua Maria Quitéria,residential,2,yes,asphalt,<NA>
3,4217299,"MULTILINESTRING ((-43.20686 -22.98095, -43.206...",Rua Alberto de Campos,residential,2,yes,asphalt,<NA>
4,4217303,"MULTILINESTRING ((-43.21387 -22.98212, -43.213...",Rua Redentor,residential,1,yes,asphalt,<NA>


## 3.2 Lendo os dados de fluxo de veículos e cálculo de ADT

In [4]:
# Lendo parquet com ADT das vias filtradas para 250 m de cada estação
roads_with_adt = pd.read_parquet(flow_path)
roads_with_adt.head()

,osm_id,weekday,hour,traffic_level,vehicle_count,vehicle_count_max,vehicle_count_min
0,4217292.00,0,0,15.85,561.90,1822.83,10.18
1,4217292.00,0,1,16.50,218.82,857.09,0.30
2,4217292.00,0,2,16.50,60.06,256.66,0.00
3,4217292.00,0,3,16.50,4.69,20.55,0.00
4,4217292.00,0,4,16.50,0.14,0.61,0.00


In [5]:
# Deixando nomes de colunas variáveis
adt_col = 'average_daily_vehicle_count'
vehicle_count_col = 'vehicle_count'

# Cálculo do ADT para cada código de via
roads_with_adt = roads_with_adt.groupby(['osm_id','weekday'])['vehicle_count'].sum()
roads_with_adt = roads_with_adt.groupby('osm_id').mean().reset_index()

# Renomeando coluna vehicle count para ADT
roads_with_adt = roads_with_adt.rename({vehicle_count_col : adt_col}, axis=1)
roads_with_adt

,osm_id,average_daily_vehicle_count
0,4217292.00,20775.76
1,4217293.00,18657.09
2,4217297.00,18560.23
3,4217299.00,20331.30
4,4217303.00,3830.52
...,...,...
359727,1385710442.00,2417.99
359728,1385866670.00,4407.47
359729,1385867601.00,0.00
359730,1385899999.00,1468.27


## 3.2 Selecionando vias com ADT calculado e > 1000 veículos

In [6]:
# Selecionando vias com ADT calculado
roads = pd.merge(roads, roads_with_adt, how='inner', on='osm_id')

In [7]:
roads.head()

,osm_id,geometry,name,highway,lanes,oneway,surface,maxspeed,average_daily_vehicle_count
0,4217292,"MULTILINESTRING ((-43.20285 -22.98436, -43.202...",Rua Vinícius de Moraes,residential,2,yes,asphalt,<NA>,20775.76
1,4217293,"MULTILINESTRING ((-43.20511 -22.98637, -43.205...",Rua Joana Angélica,residential,2,yes,asphalt,<NA>,18657.09
2,4217297,"MULTILINESTRING ((-43.20679 -22.98073, -43.206...",Rua Maria Quitéria,residential,2,yes,asphalt,<NA>,18560.23
3,4217299,"MULTILINESTRING ((-43.20686 -22.98095, -43.206...",Rua Alberto de Campos,residential,2,yes,asphalt,<NA>,20331.30
4,4217303,"MULTILINESTRING ((-43.21387 -22.98212, -43.213...",Rua Redentor,residential,1,yes,asphalt,<NA>,3830.52


In [8]:
"""Definiu-se 1000 veículos/dia como o fluxo diário médio (ADT) mínimo para uma 
via ser considerada como via principal, termo utilizado no Guia de Monitoramento
da Qualidade do Ar do Brasil."""
# Pegando vias com ADT superior a 1000 veículos/dia
roads = roads.loc[roads[adt_col] > 1000, :]
roads.head()

,osm_id,geometry,name,highway,lanes,oneway,surface,maxspeed,average_daily_vehicle_count
0,4217292,"MULTILINESTRING ((-43.20285 -22.98436, -43.202...",Rua Vinícius de Moraes,residential,2,yes,asphalt,<NA>,20775.76
1,4217293,"MULTILINESTRING ((-43.20511 -22.98637, -43.205...",Rua Joana Angélica,residential,2,yes,asphalt,<NA>,18657.09
2,4217297,"MULTILINESTRING ((-43.20679 -22.98073, -43.206...",Rua Maria Quitéria,residential,2,yes,asphalt,<NA>,18560.23
3,4217299,"MULTILINESTRING ((-43.20686 -22.98095, -43.206...",Rua Alberto de Campos,residential,2,yes,asphalt,<NA>,20331.30
4,4217303,"MULTILINESTRING ((-43.21387 -22.98212, -43.213...",Rua Redentor,residential,1,yes,asphalt,<NA>,3830.52


## 3.3 Verificando se alguma das vias não foi contemplada com dados de fluxo

In [9]:
if roads[adt_col].isna().any():
    print('Ops! Verificar!')
else:
    print('Pode seguir tranquile!')

Pode seguir tranquile!


# 4.0 Zonas Industriais

## 4.1 Lendo o arquivo de indústrias

In [10]:
""" Esta seção lê o arquivo de indústrias, com as colunas obrigatórias Longitude e Latitude"""

industrial_gdf = pd.read_csv(industrial_path)

industrial_gdf = gpd.GeoDataFrame(industrial_gdf,
                                  geometry=gpd.points_from_xy(industrial_gdf.Longitude,
                                                              industrial_gdf.Latitude,
                                                              crs='EPSG:4326'))

# Duplicando a coluna de geometria para transmitir ela após o sjoin_nearest na seção 6.2
industrial_gdf['industry_geom'] = industrial_gdf.geometry
industrial_gdf.head()

,CPF_CNPJ,NOME_PESSOA,Latitude,Longitude,ANO,CD_MUN,NM_MUN,SIGLA_UF,TIER,SETOR,...,PM25,Se,SOx,TSP,Ni,NMVOC,HCB,PCB,geometry,industry_geom
0,04.585.532/0001-99,DAX OIL REFINO S/A,-12.66,-38.31,2017,2905701,Camaçari,BA,1,Refino de petróleo,...,0.37,0.00,23.45,0.71,0.02,NaN,NaN,NaN,POINT (-38.30667 -12.65917),POINT (-38.30667 -12.65917)
1,04.585.532/0001-99,DAX OIL REFINO S/A,-12.66,-38.31,2018,2905701,Camaçari,BA,1,Refino de petróleo,...,0.40,0.00,25.72,0.78,0.03,NaN,NaN,NaN,POINT (-38.30667 -12.65917),POINT (-38.30667 -12.65917)
2,04.585.532/0001-99,DAX OIL REFINO S/A,-12.66,-38.31,2019,2905701,Camaçari,BA,1,Refino de petróleo,...,0.52,0.00,33.23,1.00,0.03,NaN,NaN,NaN,POINT (-38.30667 -12.65917),POINT (-38.30667 -12.65917)
3,04.585.532/0001-99,DAX OIL REFINO S/A,-12.66,-38.31,2020,2905701,Camaçari,BA,1,Refino de petróleo,...,0.65,0.00,41.45,1.25,0.04,NaN,NaN,NaN,POINT (-38.30667 -12.65917),POINT (-38.30667 -12.65917)
4,04.585.532/0001-99,DAX OIL REFINO S/A,-12.66,-38.31,2021,2905701,Camaçari,BA,1,Refino de petróleo,...,0.64,0.00,40.69,1.23,0.04,NaN,NaN,NaN,POINT (-38.30667 -12.65917),POINT (-38.30667 -12.65917)


In [11]:
industrial_gdf.SETOR.unique()

array(['Refino de petróleo', 'Combustão externa - indústria',
       'Produção de clínquer e cimento',
       'Indústria Química - Fertilizantes fosfatados',
       'Indústria Química - Ácido sulfúrico',
       'Indústria Química - Negro de fumo',
       'Indústria Química - Látex SBR', 'Indústria Química - HIPS',
       'Indústria Química - Estireno', 'Indústria Química - EPS',
       'Indústria Química - Polipropileno',
       'Indústria Química - Nitrato de amônio',
       'Indústria Química - Ureia', 'Indústria Química - Oxirano',
       'Indústria Química - Sulfato de amônio',
       'Indústria Química - Borracha de EB',
       'Indústria Química - Propileno', 'Indústria Química - Eteno',
       'Indústria Química - PVC', 'Produção de ferro e aço',
       'Indústria de veículos automotores - revestimento de carros',
       'Produção de Celulose e Papel'], dtype=object)

# 5.0 Estações de monitoramento da qualidade do ar

## 5.1 Lendo arquivo, determinando código EPSG e filtragem de poluentes de interesse

Esta seção faz a leitura do arquivo de estações de monitoramento da qualidade do ar 
do Brasil, com as colunas obrigatórias:
- 'LONGITUDE': float.
- 'LATITUDE': float.
- 'COD_POLUENTE': float
- 'POLUENTE': str.
- 'ID_OEMA': str.

Em seguida, cada estação é enquadrada dentro de uma zona UTM e atribui-se o código EPSG correspondente, de acordo com a zona UTM e a latitude de cada uma.

Por fim, dentre todas as linhas de estações, o geodataframe é reduzido às que monitoram os poluentes a seguir:
- monóxido de carbono (CO)
- dióxido de enxofre (SO2)
- dióxido de nitrogênio (NO2)
- ozônio (O3)
- material particulado de diâmetro inferior a 10 micrômetros (MP10)
- material particulado de diâmetro inferior a 2.5 micrômetros (MP2.5)
- material particulado total (PTS)

In [12]:
# Lendo o arquivo de estações de monitoramento
stations = pd.read_csv(filepath_or_buffer=stations_path,
                      dtype={'LONGITUDE': float,
                             'LATITUDE':float,
                             'COD_POLUENTE':float,
                             'POLUENTE':str,
                             'ID_OEMA':str
                            }
                      )

# Transformando em GeoDataFrame
stations = gpd.GeoDataFrame(stations,
                            geometry=gpd.points_from_xy(stations.LONGITUDE,
                                                        stations.LATITUDE,
                                                        crs='EPSG:4326'))
# Determinando a zona UTM para cada estação
stations.loc[:,'utm_zone'] = long_2_utm_zone(stations
                                             .geometry
                                             .centroid
                                             .x)

# Determinando do código EPSG para cada estação
stations.loc[:,'EPSG'] = utm_zone_2_epsg(stations['utm_zone'],
                                         stations.geometry
                                         .centroid
                                         .x)

# Removendo a coluna auxiliar de zona UTM
stations.drop(columns='utm_zone', inplace=True)
stations = stations[stations["UF"] != "RN"]
# Filtrando as estações que monitoram CO, SO2, O3, NO2, PM10, PM2.5 e PTS
stations = stations[stations['COD_POLUENTE'].isin([1.0, 2.0, 3.0, 4.0,
                                                   5.0, 7.0, 8.0])]

# Filtrando estações com geometrias inválidas
stations = stations[stations.geometry.is_valid & ~stations.geometry.is_empty]

/tmp/ipykernel_16881/1712969056.py:19: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .centroid
/tmp/ipykernel_16881/1712969056.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .centroid


In [13]:
stations.shape

(2245, 42)

# 6.0 Salvando outputs

In [14]:
# Vias com valores de ADT
roads.to_parquet(outputs_path / 'roads.parquet')

# Indústrias
industrial_gdf.to_parquet(outputs_path / 'industrial_gdf.parquet')

# Estações 
stations.to_parquet(outputs_path / 'stations.parquet')


In [15]:
stations.EPSG.unique()

array(['EPSG:31979', 'EPSG:31980', 'EPSG:31984', 'EPSG:31983',
       'EPSG:31982', 'EPSG:31978', 'EPSG:31981', 'EPSG:31985'],
      dtype=object)